# SELENE: Sea Level Near-real time Quality Control Processing

This notebook demonstrates how to use SELENE for quality control of tide gauge data.
Based on the Design & User's Guide v1.0 by Puertos del Estado.


## 1. Environment Setup
install selene with conda.  
We will be using the python from the conda environment in this notebook to work with selene. (/opt/conda/envs/selene_training/bin/python)  
Outside this environment you would install selene with conda and run the scripts within an activated environment.  

### 1.1 install Selene environement

In [ ]:
!chmod +x ./setup.sh
!./setup.sh

### 1.2 Install and load python packages

In [74]:
import sys 
!{sys.executable} -m pip install utide python-dotenv
print("\n✓ packages installed")

ERROR: Could not find a version that satisfies the requirement pickle (from versions: none)
ERROR: No matching distribution found for pickle

✓ packages installed


In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import configparser
import sys
import logging
import warnings
import subprocess
import requests
import matplotlib.pyplot as plt
import utide
import os
import configuration.constants as c
import ipywidgets as widgets
import csv 
import copy
from datetime import timedelta
from datetime import datetime
from dotenv import load_dotenv
from IPython.display import display, Image, JSON

print("\n✓ Imports succeeded")


✓ Imports succeeded


In [45]:
load_dotenv('./env')

True

## 2. Configuration Setup

### 2.1 Set up directory structure

In [46]:
# Define base paths (modify these according to your installation)
BASE_PATH = "./"  # Update this!
DATAFILES_PATH = os.path.join(BASE_PATH, "datafiles")
OUTPUT_PATH = os.path.join(BASE_PATH, "outputs")
CONFIG_PATH = os.path.join(BASE_PATH, "configuration")

# Create directories if they don't exist
os.makedirs(DATAFILES_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(CONFIG_PATH, exist_ok=True)

print(f"Base path: {BASE_PATH}")
print(f"Datafiles path: {DATAFILES_PATH}")
print(f"Output path: {OUTPUT_PATH}")

Base path: ./
Datafiles path: ./datafiles
Output path: ./outputs


## 3. Example Run SELENE

### 3.1 Execute SELENE Processing

In [47]:
# Target station ID
with open(c.stationsfile) as f:
    stations = json.load(f)
TARGET_STATION = widgets.Dropdown(
    options=stations.keys(),
    description='Target Station:',
    disabled=False,
)

TARGET_STATION

Dropdown(description='Target Station:', options=('2059388', '6024460', '1000000', '1000001', 'abas'), value='2…

In [111]:
# Run SELENE processing
print(f"Running SELENE for station {TARGET_STATION.value}...")
print("-" * 50)

!/opt/conda/envs/selene_training/bin/python selene.py {TARGET_STATION.value} 

Running SELENE for station abas...
--------------------------------------------------
/opt/conda/envs/selene_training/lib/python3.8/site-packages/utide/harmonics.py:16: RuntimeWarning: invalid value encountered in cast
  nshallow = np.ma.masked_invalid(const.nshallow).astype(int)
/opt/conda/envs/selene_training/lib/python3.8/site-packages/utide/harmonics.py:17: RuntimeWarning: invalid value encountered in cast
  ishallow = np.ma.masked_invalid(const.ishallow).astype(int) - 1


### 3.2 Buddy Check Processing

In [112]:
# Buddy check requires target year as last year of data
TARGET_YEAR = widgets.IntSlider(
    value=2026,
    min=1970,
    max=2026,
    step=1,
    description='TARGET YEAR:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

TARGET_YEAR

IntSlider(value=2026, continuous_update=False, description='TARGET YEAR:', max=2026, min=1970)

In [ ]:
print(f"Running buddy check for station {TARGET_STATION.value}, year {TARGET_YEAR.value}...")

!/opt/conda/envs/selene_training/bin/python buddy_check.py {TARGET_STATION.value} {TARGET_YEAR.value}

Running buddy check for station abas, year 2026...
/opt/conda/envs/selene_training/lib/python3.8/site-packages/utide/harmonics.py:16: RuntimeWarning: invalid value encountered in cast
  nshallow = np.ma.masked_invalid(const.nshallow).astype(int)
/opt/conda/envs/selene_training/lib/python3.8/site-packages/utide/harmonics.py:17: RuntimeWarning: invalid value encountered in cast
  ishallow = np.ma.masked_invalid(const.ishallow).astype(int) - 1


### 3.3 Visualization

#### 3.3.1 Generate the graphs

In [ ]:
# Run SELENEVIS for visualization
print(f"Running SELENEVIS for station {TARGET_STATION.value}...")

!/opt/conda/envs/selene_training/bin/python selenevis.py -station {TARGET_STATION.value}

#### 3.3.2 Open the generated image

In [ ]:
with open(c.stationsfile) as f:
    stations = json.load(f)
    station_name = stations[TARGET_STATION.value]['name']
    
Image(filename=OUTPUT_PATH + "/SELENEQC_graphics_Station_" + station_name + ".png")

#### 3.3.3 View Output files for targeted station

In [33]:
# List output files
print( OUTPUT_PATH)
output_files = []
if os.path.exists('./' + OUTPUT_PATH):
    output_files = os.listdir('./' + OUTPUT_PATH)
    
for file in sorted(output_files):
    if TARGET_STATION.value in file:
        filepath = os.path.join(OUTPUT_PATH, file)
        size = os.path.getsize(filepath)
        print(f"  - {file} ({size:,} bytes)")

./outputs


## 4. Custom Data Integration

To add your own station data:

1. **Prepare data file**: Create ASCII .data file in `datafiles/`
2. **Update stations.json**: Add station configuration
3. **Update code_guide.csv** (if using): Add station metadata
4. **Run SELENE**: Execute with your station ID

## 4.1 Retrieve data from SLSMF

## SLSMF API 
### Authentication: 
Go to our website and request an account [here](https://ioc-sealevelmonitoring.org/api.php).  
After your account has been approved you can request an api key [here](https://ioc-sealevelmonitoring.org/api.php).  
Place your api key in the env file, in accordance to the env.example file, or place copy it below.
### Documentation
Go to [api documentation](https://api.ioc-sealevelmonitoring.org/v2/doc), for a complete overview of our endpoints.  
If you wish to try out some enpoints use the api key you just created.  


#### Set API_KEY
store your api key in the ./env file or provide it below

In [25]:
print(os.getenv('API_KEY'))
APIKEY = widgets.Text(
    value=os.getenv('API_KEY'),
    placeholder='Type something',
    description='APIKEY:',
    disabled=False   
)
APIKEY

bd423717a30e8414189510b75eb36655049d3e1341d86b9b99e83eec2210534ea6f5bd4ecf37f447ec41588f12c1cb871bc6669dfaef758c213e6b522a078d47


Text(value='bd423717a30e8414189510b75eb36655049d3e1341d86b9b99e83eec2210534ea6f5bd4ecf37f447ec41588f12c1cb871b…

#### Download the Real Time data 
##### Set the api parameters

In [26]:
station = widgets.Text(
    value='abas',
    placeholder='IOC code',
    description='IOC:',
    disabled=False   
)
sensor = widgets.Text(
    value='rad',
    description='sensor type:',
    disabled=False   
)
timestart=widgets.DatePicker(
    description='timestart',
    disabled=False
)
timestop=widgets.DatePicker(
    description='timestop',
    disabled=False
)

from ipywidgets import Box

items = [station, sensor, timestart, timestop]
box = Box(children=items)

box

Box(children=(Text(value='abas', description='IOC:', placeholder='IOC code'), Text(value='rad', description='s…

##### do the download 
we can only request 30 days so we need to loop this

In [35]:
url = "https://api.ioc-sealevelmonitoring.org/v2/stations/"+ station.value
querystring = {}
headers = {"X-API-KEY": os.getenv('API_KEY'), "accept": "aplication/json"}
response = requests.get(url, headers=headers, params=querystring)
station_metadata = json.loads(response.text)[0]
print(response.url)
print(station_metadata)

https://api.ioc-sealevelmonitoring.org/v2/stations/abas
{'Code': 'abas', 'Location': 'Abashiri', 'country': 'JAP', 'type': 'SL', 'DCP_ID': 'ABASHIRI', 'WMO': 'SWJP40', 'XMtInt': 10, 'Lat': 44.02, 'Lon': 144.29, 'Har': 0, 'GlossID': '327 &nbsp;&nbsp;<a href="https://www.bodc.ac.uk/resources/inventories/gloss_handbook/stations/327/" target="_blank">[goto handbook]</a>', 'OT12Code': None, 'Example': None, 'UTCOffset': 0, 'OperatorID': None, 'status': 'Operational', 'status_description': None, 'localoperator': 'Japan Meteorological Agency ( Japan )', 'performanceoperator1': None, 'performanceoperator2': None, 'performanceoperator3': None, 'date_created': '2012-03-21 09:54:59', 'url': None, 'statsensor': 1406, 'updatedata': '8', 'timestampdata': '7', 'timestampupdate': '0', 'transmittype': 'GTS message', 'countryname': 'Japan', 'example_message': 'SWJP40 RJTD 071220\r\r\nLAT   LONG   TIME    STATION\r\r\n4298N 14437E 1271219 KUSHIRO\r\r\nLEVELS\r\r\n0740 0744 0742 0742 0743 0742 0740 0739 0

In [105]:
if timestart.value and timestop.value: 
    loop_start = copy.deepcopy(timestart.value)
    output_file = DATAFILES_PATH + '/' + station.value + '.data' 
    if os.path.exists(output_file):
        os.remove(output_file)
        
    while (timestop.value - loop_start).total_seconds() > 0:
        timestop_parameter = loop_start + timedelta(days=30) 
        timestop_parameter = timestop.value if timestop_parameter > timestop.value else timestop_parameter
        url = "https://api.ioc-sealevelmonitoring.org/v2/stations/"+ station.value + "/data"
        querystring = {"timestart": loop_start.strftime("%Y-%m-%d"), "timestop": timestop_parameter  }
        headers = {"X-API-KEY": os.getenv('API_KEY'), "accept": "application/json"}
        response = requests.get(url, headers=headers, params=querystring)
        print(response.url)
        file = DATAFILES_PATH + '/' + station.value + '.data'
        with open(file, "a") as f:
            for entry in json.loads(response.text):
                slevel = float(entry['slevel'])*1000
                stime = entry['stime']
                f.write(f'{station.value} {stime} {slevel} 1\n')
        loop_start = timestop_parameter
    print("\n Download complete, data stored at " + file)
else:
    print("\nPlease provide timestart and timestop in the cell above. ")

https://api.ioc-sealevelmonitoring.org/v2/stations/abas/data?timestart=2025-05-07&timestop=2025-06-06
https://api.ioc-sealevelmonitoring.org/v2/stations/abas/data?timestart=2025-06-06&timestop=2025-07-06
https://api.ioc-sealevelmonitoring.org/v2/stations/abas/data?timestart=2025-07-06&timestop=2025-08-05
https://api.ioc-sealevelmonitoring.org/v2/stations/abas/data?timestart=2025-08-05&timestop=2025-09-04
https://api.ioc-sealevelmonitoring.org/v2/stations/abas/data?timestart=2025-09-04&timestop=2025-10-04
https://api.ioc-sealevelmonitoring.org/v2/stations/abas/data?timestart=2025-10-04&timestop=2025-11-03
https://api.ioc-sealevelmonitoring.org/v2/stations/abas/data?timestart=2025-11-03&timestop=2025-12-03
https://api.ioc-sealevelmonitoring.org/v2/stations/abas/data?timestart=2025-12-03&timestop=2026-01-02
https://api.ioc-sealevelmonitoring.org/v2/stations/abas/data?timestart=2026-01-02&timestop=2026-02-01
https://api.ioc-sealevelmonitoring.org/v2/stations/abas/data?timestart=2026-02-01&

In [45]:
import csv
import io
import os
from urllib.parse import urljoin

import requests


if timestart.value and timestop.value:
    output_file = os.path.join(DATAFILES_PATH, f"{station.value}.data")

    if os.path.exists(output_file):
        os.remove(output_file)

    base_url = (
        f"https://api.ioc-sealevelmonitoring.org/v2/research/"
        f"stations/{station.value}/sensors/one-sensor/data"
    )

    headers = {
        "accept": "text/csv",
    }

    # Only add API key if you still need it for this endpoint.
    api_key = os.getenv("API_KEY")
    if api_key:
        headers["X-API-KEY"] = api_key

    params = {
        "days_per_page": 365,
        "page": 1,
        "timestart": timestart.value.strftime("%Y-%m-%d"),
        "timestop": timestop.value.strftime("%Y-%m-%d"),
        "subtract_30d_average": "false",
        "flag_qc": "false",
        "filter_completeness": "true",
        "filter_distinctness": "true",
        "filter_shift": "true",
        "filter_out_of_range": "true",
        "filter_exceeded_neighbours": "true",
        "filter_spikes_via_median": "true",
        "filter_flat_line": "true",
        "fit_to_sample_rate": "false",
    }

    page = 1
    total_pages = None

    with open(output_file, "a", newline="") as f:
        writer = csv.writer(
            f,
            delimiter=" ",
            quotechar='"',
            quoting=csv.QUOTE_MINIMAL,
        )

        while total_pages is None or page <= total_pages:
            print(total_pages)
            params["page"] = page

            response = requests.get(
                base_url,
                headers=headers,
                params=params,
                timeout=60,
            )
            response.raise_for_status()

            print(response.url)

            lines = response.text.strip().splitlines()

            if len(lines) < 4:
                print(f"Warning: no data returned for page {page}")

            # Response layout:
            # line 0: pagination header
            # line 1: pagination values
            # line 2: data header
            # line 3+: data rows
            pagination_header = lines[0].split("\t")
            pagination_values = lines[1].split("\t")
            pagination = dict(zip(pagination_header, pagination_values))

            current_page = int(pagination.get("current_page", page))
            total_pages = int(pagination.get("total_pages", current_page))
            next_page = pagination.get("next_page")

            data_text = "\n".join(lines[2:])
            reader = csv.DictReader(io.StringIO(data_text), delimiter="\t")

            for entry in reader:
                slevel = float(entry.get("slevel")) * 1000 if entry.get("slevel") != 'NA' else 'NA'
                stime = entry.get("stime")

                if not stime or slevel in (None, ""):
                    continue

                    

                writer.writerow([
                    station.value,
                    stime,
                    slevel,
                    1,
                ])

            if not next_page:
                break

            page = int(next_page)

    print(f"\nDownload complete, data stored at {output_file}")

else:
    print("\nPlease provide timestart and timestop in the cell above.")

None
https://api.ioc-sealevelmonitoring.org/v2/research/stations/abas/sensors/one-sensor/data?days_per_page=365&page=1&timestart=2024-01-01&timestop=2026-01-01&subtract_30d_average=false&flag_qc=false&filter_completeness=true&filter_distinctness=true&filter_shift=true&filter_out_of_range=true&filter_exceeded_neighbours=true&filter_spikes_via_median=true&filter_flat_line=true&fit_to_sample_rate=false
3
https://api.ioc-sealevelmonitoring.org/v2/research/stations/abas/sensors/one-sensor/data?days_per_page=365&page=2&timestart=2024-01-01&timestop=2026-01-01&subtract_30d_average=false&flag_qc=false&filter_completeness=true&filter_distinctness=true&filter_shift=true&filter_out_of_range=true&filter_exceeded_neighbours=true&filter_spikes_via_median=true&filter_flat_line=true&fit_to_sample_rate=false
3
https://api.ioc-sealevelmonitoring.org/v2/research/stations/abas/sensors/one-sensor/data?days_per_page=365&page=3&timestart=2024-01-01&timestop=2026-01-01&subtract_30d_average=false&flag_qc=false

In [106]:
names = ["station", "date", "time", "value", "flag"]

obs = pd.read_csv(
    DATAFILES_PATH + '/' + station.value + '.data',
    names=names,
    skipinitialspace=True,
    sep='\s+',    
    na_values="9.999",
)

obs["anomaly"] = obs["value"] - obs["value"].mean()
index = pd.to_datetime(obs["date"] + ' ' + obs['time'])
obs.index = index
obs.dropna(subset=["value", "anomaly"], inplace=True)
mean = obs["value"].mean()
obs.head(5)


,station,date,time,value,flag,anomaly
2025-05-07 00:00:00,abas,2025-05-07,00:00:00,2142.7,1,122.512194
2025-05-07 00:01:00,abas,2025-05-07,00:01:00,2142.7,1,122.512194
2025-05-07 00:02:00,abas,2025-05-07,00:02:00,2151.9,1,131.712194
2025-05-07 00:03:00,abas,2025-05-07,00:03:00,2154.9,1,134.712194
2025-05-07 00:04:00,abas,2025-05-07,00:04:00,2154.9,1,134.712194


In [68]:

import datetime as dt

years = list({i for i in obs.index.year})
coeffs = {}
for year in years: 
    include = obs[obs.index.year == year]
    years = list({i for i in include.index.year})
    print(f'calculating coefs for {year}')
    print(datetime.now())
    coef = utide.solve(
        include.index,
        include["anomaly"],
        lat=44.02,
        method="ols",
        conf_int="MC",
        verbose=False
    )
    
    print(datetime.now())


calculating coefs for 2026
2026-05-07 14:34:34.370794


LinAlgError: SVD did not converge

In [109]:
import io   
url = f"https://api.ioc-sealevelmonitoring.org/v2/research/stations/{station.value}/sensors/rad/tidal-harmonics"
querystring = {"lateral_correction":"true"}
headers = {"X-API-KEY": os.getenv('API_KEY'), "accept": "text/csv"}
response = requests.get(url, headers=headers, params=querystring)
SLSMF_coeffs = response.text 
print(response.url)
print(f"coeffs retrieved for {station.value}")


https://api.ioc-sealevelmonitoring.org/v2/research/stations/abas/sensors/rad/tidal-harmonics?lateral_correction=true
coeffs retrieved for abas


In [110]:
import io
from datetime import datetime

import numpy as np
import pandas as pd
import pickle 
from utide._ut_constants import constit_index_dict
from utide.utilities import Bunch
from utide._time_conversion import _python_gregorian_datenum
from utide import reconstruct


df = pd.read_csv(io.StringIO(SLSMF_coeffs), sep=r"\s+", skiprows=2, engine="python")

SLSMF_coeffs_metadata = "\n".join(SLSMF_coeffs.split("\n")[0:2])
df_metadata = pd.read_csv(io.StringIO(SLSMF_coeffs_metadata), sep=r"\s+", engine="python")

coeffs_time_reference = datetime.strptime(
    df_metadata.iloc[0]["time_reference"],
    "%Y-%m-%d",
)

# Handle Z0.
z0_row = df[df["name"] == "Z0"]

if z0_row.empty:
    z0_value = 0.0
    print("Warning: No 'Z0' constituent found. Setting mean sea level to 0.")
else:
    z0_value = float(z0_row["amplitude"].iloc[0])

df = df[df["name"] != "Z0"].copy()

constituents = df["name"].astype(str).to_numpy()
amplitudes = df["amplitude"].astype(float).to_numpy()
phases_deg = df["phase"].astype(float).to_numpy()
frequencies = df["frequency"].astype(float).to_numpy()  # cycles/hour

# IMPORTANT:
# Mongo computes A*cos(omega*t - phase), so UTide g should be phase as-is.
phases_for_utide_g = phases_deg

missing = [nm for nm in constituents if nm not in constit_index_dict]
if missing:
    raise ValueError(f"These constituents are not known by UTide: {missing}")

lind = np.array([constit_index_dict[nm] for nm in constituents], dtype=int)

oce2utide_coef = Bunch(
    name=constituents,
    A=amplitudes,
    g=phases_for_utide_g,
    A_ci=np.zeros(len(constituents)),
    g_ci=np.zeros(len(constituents)),
    mean=z0_value ,
    slope=0.0,
    aux=Bunch(
        reftime=float(_python_gregorian_datenum(coeffs_time_reference)),
        lat=float(station_metadata["Lat"]),
        frq=frequencies,
        lind=lind,
        opt=Bunch(
            twodim=False,
            notrend=True,
            nodiagn=True,

            # No astronomical/nodal conversion.
            # Use the OCE table directly relative to time_reference.
            nodsatlint=False,
            nodsatnone=True,
            gwchlint=False,
            gwchnone=True,

            prefilt=[],
            conf_int="none",
        ),
    ),
)

years = list({i for i in obs.index.year})
coeffs = {}
for year in years: 
    coeffs[year] = oce2utide_coef
with open(f'{CONFIG_PATH}/harmonics/{station.value}_harm_all.pkl', 'wb') as handle:
    pickle.dump(coeffs, handle, protocol=pickle.HIGHEST_PROTOCOL)

import json
import numpy as np
from utide.utilities import Bunch


def to_serializable(obj):
    if isinstance(obj, Bunch):
        return {key: to_serializable(value) for key, value in obj.items()}

    if isinstance(obj, dict):
        return {key: to_serializable(value) for key, value in obj.items()}

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        return float(obj)

    if isinstance(obj, (list, tuple)):
        return [to_serializable(value) for value in obj]

    return obj


output_txt = "oce2utide_coef.txt"

coeffs_str = {}

for  key in coeffs.keys():
    coeffs_str[key] = to_serializable(coeffs[key]),


with open(f'{CONFIG_PATH}/harmonics/{station.value}_harm_all.txt', "w") as f:
    json.dump(
        coeffs_str,
        f,
        indent=4,
    )



In [49]:
  
# Load existing configuration
with open(c.stationsfile, 'r') as f:
    stations = json.load(f)

# Add new station
stations[station_metadata['Code']] = {
    "foremanharmfile": f"configuration/harmonics/{station.value}_harm.pkl",
    "latitude": station_metadata['Lat'],
    "longitude": station_metadata['Lon'],
    "minint": 60,
    "maxlevel": 18406,
    "maxsurge": 13497,
    "minlevel": -8588,
    "minsurge": -13497,
    "max": 2300,
    "qc_level_nsigma": 3,
    "qc_level_splinedegree": 2,
    "qc_level_winsize": 40,
    "qc_stucklimit": 10,
    "qc_surge_nsigma": 5,
    "qc_surge_splinedegree": 3,
    "qc_surge_winsize": 80,
    "shortname": station_metadata['Code'].upper(),
    "name": station_metadata['Code'],
    "seriesdatecolumns": "1,2",
    "seriesdateformat": "%Y-%m-%d%H:%M:%S",
    "seriesfile": f"datafiles/{station.value + '.data'}",
    "seriesqccolumn": 4,
    "seriesseparator": " ",
    "seriesvaluecolumn": 3,
    "data_mode": "R",
    }

# Save updated configuration
with open(c.stationsfile, 'w') as f:
    json.dump(stations, f, indent=4)

print(f"✓ Added station {station_metadata['Code']}")


display(JSON(c.stationsfile, expanded=False))


✓ Added station abas


<IPython.core.display.JSON object>

## 8. Troubleshooting

### Common Issues

**1. Module not found errors**
- Ensure you're in the SELENE directory
- Check that Python environment is activated: `conda activate selene_training`

**2. Permission denied**
- Give execution permissions: `chmod 755 foreman/predicc.e`

**3. Data format errors**
- Verify .data file format matches specifications
- Check column separators and date/time formats

**4. Missing harmonic constants**
- Set `foremanharmfile` to empty string ("") if not available
- Tide/surge analysis will be skipped

### Runtime Warnings

You may see warnings like:
```
RankWarning: Polyfit may be poorly conditioned
```
These are expected and don't prevent processing from completing.

## 9. Output Files Reference

SELENE generates several output files in the `output/` directory:

| Algorithm | Output Files | Description |
|-----------|--------------|-------------|
| Selene.py | `{code}_original_sampling_flags.out` | Original data with quality flags |
| Buddy_check.py | `{code}_original_sampling_buddy.out` | Buddy check results |
| | `{code}_hourly_slev_buddy.out` | Hourly sea levels |
| | `{code}_hourly_surge_buddy.out` | Hourly surges |
| | `{code}_hourly_tide_buddy.out` | Hourly tides |
| Selenevis.py | `{station}_*.png` | Visualization plots |

## 10. Next Steps

### Additional Resources

- **Official Documentation**: https://puertos-del-estado-medio-fisico.github.io/SELENE/
- **Copernicus Marine Service**: https://data.marine.copernicus.eu/product/INSITU_GLO_PHY_SSH_DISCRETE_MY_013_053/description
- **Support Contact**: Dr. Begoña Pérez Gómez - bego@puertos.es

### Advanced Usage

- **Automated Processing**: Set up cron jobs with `selenelauncher.py`
- **Database Integration**: Use `selenedatadownload.py` and `selenedbconsolide.py`
- **Custom Filters**: Modify `filterhandler.py` for different filtering methods
- **Parallel Processing**: Configure multiprocessing in launcher script

---

**Note**: This notebook is based on SELENE Version 1.0 by Puertos del Estado.
For production use, ensure all configuration parameters are properly set for your
specific stations and data characteristics.

# Getting data from SLSMF
## SLSMF API 
### Authentication: 
Go to our website and request an account [here](https://ioc-sealevelmonitoring.org/api.php).  
After your account has been approved you can request an api key [here](https://ioc-sealevelmonitoring.org/api.php).  
### Documentation
Go to [api documentation](https://api.ioc-sealevelmonitoring.org/v2/doc), for a complete overview of our endpoints.  
If you wish to try out some enpoints use the api key you just created.  

In [ ]:



# Load environment variables from the .env file
load_dotenv('./env')

url = "https://api.ioc-sealevelmonitoring.org/v2/research/stations/abas/sensors/rad/tidal-harmonics"
querystring = {"datestop": "2026-05-06", "lateral_correction":"true"}
headers = {"X-API-KEY": os.getenv('API_KEY'), "accept": "text/csv"}
response = requests.get(url, headers=headers, params=querystring)

print(response.text)

In [ ]:

# Load environment variables from the .env file
load_dotenv('./env')

url = "https://api.ioc-sealevelmonitoring.org/v2/stations/abas/data"
querystring = {}
headers = {"X-API-KEY": os.getenv('API_KEY'), "accept": "text/csv"}
response = requests.get(url, headers=headers, params=querystring)

print(response.text)

# Utide intecration 

In [100]:
print("\n" + "="*60)
print("Notebook completed!")
print("="*60)
print("\nTo run SELENE with your data:")
print(f"  1. Place your .data file in: {DATAFILES_PATH}")
print(f"  2. Update stations.json with your station configuration")
print(f"  3. Run: /opt/conda/envs/selene_training/bin/python selene.py YOUR_STATION_ID")
print(f"  4. Visualize: /opt/conda/envs/selene_training/bin/python selenevis.py -station YOUR_STATION_ID")


Notebook completed!

To run SELENE with your data:
  1. Place your .data file in: ./datafiles
  2. Update stations.json with your station configuration
  3. Run: /opt/conda/envs/selene_training/bin/python selene.py YOUR_STATION_ID
  4. Visualize: /opt/conda/envs/selene_training/bin/python selenevis.py -station YOUR_STATION_ID


In [101]:
import sys 
!{sys.executable} -m pip install utide

In [102]:
%matplotlib inline


print(utide.__version__)

0.3.1


Look at the data file to see what structure it has.

In [103]:
datafile = os.path.join(DATAFILES_PATH, "can1998.dtf")
with open(datafile) as f:
    lines = f.readlines()

print("".join(lines[:5]))

         0 1998  1  1  0.0000     1.200 0
      3600 1998  1  1  1.0000     1.430 0
      7200 1998  1  1  2.0000     1.730 0
     10800 1998  1  1  3.0000     2.030 0
     14400 1998  1  1  4.0000     2.380 0



It looks like the fields are seconds, year, month, day, hour, elevation, flag.  We need a date parser function to combine the date and time fields into a single value to be used as the datetime index.

In [104]:
names = ["seconds", "year", "month", "day", "hour", "elev", "flag"]

obs = pd.read_csv(
    datafile,
    names=names,
    skipinitialspace=True,
    delim_whitespace=True,
    na_values="9.990",
)


date_cols = ["year", "month", "day", "hour"]
index = pd.to_datetime(obs[date_cols])
obs = obs.drop(date_cols, axis=1)
obs.index = index

obs.head(5)

/tmp/ipykernel_187/1863008529.py:3: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  obs = pd.read_csv(


,seconds,elev,flag
1998-01-01 00:00:00,0,1.20,0
1998-01-01 01:00:00,3600,1.43,0
1998-01-01 02:00:00,7200,1.73,0
1998-01-01 03:00:00,10800,2.03,0
1998-01-01 04:00:00,14400,2.38,0


Although there are no elevations marked bad via special value, which should be `nan` after reading the file, the flag value of 2 indicates the values are unreliable, so we will mark them with `nan`, calculate the deviations of the elevations from their mean (stored in a new column called "anomaly"), and then interpolate to fill in the `nan` values in the anomaly.

In [108]:
bad = obs["flag"] == 2
corrected = obs["flag"] == 1

obs.loc[bad, "elev"] = np.nan
obs["anomaly"] = obs["elev"] - obs["elev"].mean()
obs["anomaly"] = obs["anomaly"].interpolate()
print(f"{bad.sum()} points were flagged 'bad' and interpolated")
print(f"{corrected.sum()} points were flagged 'corrected' and left unchanged")
obs.head(5)

10 points were flagged 'bad' and interpolated
212 points were flagged 'corrected' and left unchanged


,seconds,elev,flag,anomaly
1998-01-01 00:00:00,0,1.20,0,-0.569914
1998-01-01 01:00:00,3600,1.43,0,-0.339914
1998-01-01 02:00:00,7200,1.73,0,-0.039914
1998-01-01 03:00:00,10800,2.03,0,0.260086
1998-01-01 04:00:00,14400,2.38,0,0.610086


Now we can call solve to obtain the coefficients.

In [106]:
coef = utide.solve(
    obs.index,
    obs["anomaly"],
    lat=-25,
    method="ols",
    conf_int="MC",
    verbose=False,
)

The amplitudes and phases from the fit are now in the `coef` data structure (a Bunch), which can be used directly in the `reconstruct` function to generate a hindcast or forecast of the tides at the times specified in the `time` array.

In [107]:
print(coef)

A       : [3.71220478e-01 2.42991812e-01 1.09297864e-01 7.73350128e-02
 6.77366585e-02 6.51779557e-02 6.37862484e-02 5.67144408e-02
 3.64668856e-02 3.24316707e-02 3.14847542e-02 3.11233442e-02
 2.74651335e-02 2.73697794e-02 2.65278834e-02 2.64884084e-02
 2.62671383e-02 2.58648599e-02 2.52725340e-02 2.26786092e-02
 1.98222403e-02 1.58818273e-02 1.33720232e-02 1.05771748e-02
 9.72959564e-03 9.51812745e-03 8.19192192e-03 7.24969099e-03
 6.92317214e-03 6.77220826e-03 6.11266445e-03 5.16968038e-03
 4.83224711e-03 4.81055489e-03 3.43796408e-03 3.19281780e-03
 3.19168992e-03 3.12285084e-03 3.11619748e-03 2.92108211e-03
 2.90354584e-03 2.89918123e-03 2.70040522e-03 2.53299528e-03
 2.26569813e-03 2.02602526e-03 1.81924596e-03 1.61571235e-03
 1.60763133e-03 1.51070964e-03 1.43566416e-03 1.43013412e-03
 1.14563805e-03 9.02565682e-04 8.72012450e-04 8.32881325e-04
 5.64913869e-04 5.38638406e-04 2.56227282e-04]
A_ci    : [0.00244933 0.00224469 0.00235128 0.0029309  0.00303608 0.00222105
 0.00243271 

In [ ]:
tide = utide.reconstruct(obs.index, coef, verbose=False)

The output from the reconstruction is also a Bunch:

In [ ]:
print(tide.keys())

In [ ]:
t = obs.index.to_pydatetime()

fig, (ax0, ax1, ax2) = plt.subplots(figsize=(17, 5), nrows=3, sharey=True, sharex=True)

ax0.plot(t, obs.anomaly, label="Observations", color="C0")
ax1.plot(t, tide.h, label="Prediction", color="C1")
ax2.plot(t, obs.anomaly - tide.h, label="Residual", color="C2")
fig.legend(ncol=3, loc="upper center");